In [14]:
import sys

In [15]:
#!{sys.executable} -m pip install langchain-openai langchain-community
!{sys.executable} -m pip install faiss-cpu

  Using cached faiss_cpu-1.13.2-cp310-abi3-macosx_14_0_arm64.whl.metadata (7.6 kB)
Using cached faiss_cpu-1.13.2-cp310-abi3-macosx_14_0_arm64.whl (3.5 MB)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [8]:
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
import pandas as pd

In [2]:
load_dotenv("si699.env")

True

## Part 1 Chunking

In [9]:
damage_df = pd.read_json("dnd5eapi_dump/damage-types.json")
damage_df

,endpoint,endpoint_url,items
0,damage-types,https://www.dnd5eapi.co/api/2014/damage-types,"{'index': 'acid', 'name': 'Acid', 'desc': ['Th..."
1,damage-types,https://www.dnd5eapi.co/api/2014/damage-types,"{'index': 'bludgeoning', 'name': 'Bludgeoning'..."
2,damage-types,https://www.dnd5eapi.co/api/2014/damage-types,"{'index': 'cold', 'name': 'Cold', 'desc': ['Th..."
3,damage-types,https://www.dnd5eapi.co/api/2014/damage-types,"{'index': 'fire', 'name': 'Fire', 'desc': ['Re..."
4,damage-types,https://www.dnd5eapi.co/api/2014/damage-types,"{'index': 'force', 'name': 'Force', 'desc': ['..."
5,damage-types,https://www.dnd5eapi.co/api/2014/damage-types,"{'index': 'lightning', 'name': 'Lightning', 'd..."
6,damage-types,https://www.dnd5eapi.co/api/2014/damage-types,"{'index': 'necrotic', 'name': 'Necrotic', 'des..."
7,damage-types,https://www.dnd5eapi.co/api/2014/damage-types,"{'index': 'piercing', 'name': 'Piercing', 'des..."
8,damage-types,https://www.dnd5eapi.co/api/2014/damage-types,"{'index': 'poison', 'name': 'Poison', 'desc': ..."
9,damage-types,https://www.dnd5eapi.co/api/2014/damage-types,"{'index': 'psychic', 'name': 'Psychic', 'desc'..."


In [10]:
chunks = []
for i in range(len(damage_df)):
    endpoint, url, items = damage_df.iloc[i, :]
    chunk = {}
    chunk["chunk_id"] = "damage_" + items["index"]
    chunk["text"] = f"""Damage_type: {items["name"]}
    Description: {" ".join(items["desc"])}""".strip()
    chunk["metadata"]={
            "endpoint": endpoint,
            "endpoint_url": url,
            "index": items["index"],
            "name": items["name"],
            "updated_at": items["updated_at"],
            "url": items["url"]
        }
    chunks.append(chunk)

## Part2: Embedding

In [16]:
texts = [chunk["text"] for chunk in chunks]
metadatas = [chunk["metadata"] | {"chunk_id": chunk["chunk_id"]} for chunk in chunks]

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = FAISS.from_texts(
    texts=texts,
    embedding=embedding_model,
    metadatas=metadatas
)


In [17]:
vectorstore.save_local("faiss_damage_index")

## Part 3: Retrival

In [18]:
def retrieve_chunks(vectorstore, query, k=3):
    results = vectorstore.similarity_search(query, k=k)
    return results

In [19]:
def build_context(retrieved_docs):
    context_parts = []
    for i, doc in enumerate(retrieved_docs, 1):
        chunk_text = doc.page_content
        source_name = doc.metadata.get("name", "Unknown")
        chunk_id = doc.metadata.get("chunk_id", "N/A")

        context_parts.append(
            f"[Chunk {i} | {chunk_id} | {source_name}]\n{chunk_text}"
        )

    return "\n\n".join(context_parts)

In [20]:
rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a helpful Dungeons & Dragons rules assistant.

Use only the retrieved context below to answer the question.
If the answer is not contained in the context, say "I don't know based on the retrieved context."

Retrieved Context:
{context}

Question:
{question}

Answer:
""".strip()
)

In [21]:
def generate_answer(llm, prompt_template, context, question):
    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )
    response = llm.invoke(formatted_prompt)
    return response.content

In [22]:
def rag_pipeline(query, vectorstore, llm, prompt_template, k=3):
    retrieved_docs = retrieve_chunks(vectorstore, query, k=k)
    context = build_context(retrieved_docs)
    answer = generate_answer(llm, prompt_template, context, query)

    return {
        "question": query,
        "retrieved_docs": retrieved_docs,
        "context": context,
        "answer": answer
    }

In [23]:
llm = ChatOpenAI(model="gpt-4o-mini")

query = "What damage type is associated with a blue dragon's breath?"

result = rag_pipeline(query, vectorstore, llm, rag_prompt, k=3)

print("Question:")
print(result["question"])
print("\nRetrieved Context:")
print(result["context"])
print("\nAnswer:")
print(result["answer"])

Question:
What damage type is associated with a blue dragon's breath?

Retrieved Context:
[Chunk 1 | damage_lightning | Lightning]
Damage_type: Lightning
    Description: A lightning bolt spell and a blue dragon's breath deal lightning damage.

[Chunk 2 | damage_poison | Poison]
Damage_type: Poison
    Description: Venomous stings and the toxic gas of a green dragon's breath deal poison damage.

[Chunk 3 | damage_fire | Fire]
Damage_type: Fire
    Description: Red dragons breathe fire, and many spells conjure flames to deal fire damage.

Answer:
The damage type associated with a blue dragon's breath is lightning damage.
